In [1]:
!pip install sentence-transformers chromadb groq pandas -q
print("Installation completed")

Installation completed


In [2]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All libraries imported successfully")

All libraries imported successfully


In [3]:
GROQ_API_KEY = "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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq API  client initialized.")
print("Note: If you see an authentication error later")

Groq API  client initialized.
Note: If you see an authentication error later


In [4]:
df = pd.read_csv('college_notes.csv')
print("Shape of dataset", df.shape)
print("\nColumn names", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Shape of dataset (14, 4)

Column names ['note_id', 'subject', 'topic', 'content']

First 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [5]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())
print("\nSample of topics:")
print(df[['note_id', 'subject', 'topic']].head(3).to_string(index=False))
print("\nLength of content (no. of char) for each note:")
df['content_length'] = df['content'].apply(len)
print(df[['topic', 'content_length']].head(3).to_string(index=False))

Subjects in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    1
Name: count, dtype: int64

Sample of topics:
note_id          subject         topic
   N001 Data Engineering ETL Pipelines
   N002 Data Engineering SQL Databases
   N003 Data Engineering Data Cleaning

Length of content (no. of char) for each note:
        topic  content_length
ETL Pipelines             216
SQL Databases             209
Data Cleaning             210


In [6]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']} " for row in df.to_dict('records')]
metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared : {len(documents)}")
print(f"First document Id : {ids[0]}")
print(f"First metadata : {metadatas[0]}")
print(f"First 100 chars of doc:{documents[0][:100]}...")

Total chunks prepared : 14
First document Id : note_N001 
First metadata : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [7]:
print("Loading embedding model")
print("This may take 30-60 seconds")
print("(Subsequent runs will be faster as the model is cached)}")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("\nEmbedding model loaded successfully")
test_embedding = embedding_model.encode("This is a test sentence")
print(f"Test embedding shape:{test_embedding.shape}")
print(f"First 5 values of test embedding:{test_embedding[:5]}")

Loading embedding model
This may take 30-60 seconds
(Subsequent runs will be faster as the model is cached)}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding model loaded successfully
Test embedding shape:(384,)
First 5 values of test embedding:[0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [8]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name = "college_notes_rag")
print("ChromaDB client created")
print(f"Collection name:College_notes_rag")
print(f"Documetns in collection so far: {collection.count()}")

ChromaDB client created
Collection name:College_notes_rag
Documetns in collection so far: 0


In [9]:
print("Generating embedding for all the 15 notes")
print("This may take 15-30 sec")
embeddings = embedding_model.encode(documents, show_progress_bar= True)
print(f"\n Embedding matrix shape: {embeddings.shape}")
embeddings_list = embeddings.tolist()
collection.add(
    documents = documents,
    embeddings = embeddings_list,
    ids = ids,
    metadatas = metadatas
)
print(f"\nDocuments successfully added to chromadb.")
print(f"\nTotal documents in collection:{collection.count()}")

Generating embedding for all the 15 notes
This may take 15-30 sec


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


 Embedding matrix shape: (14, 384)

Documents successfully added to chromadb.

Total documents in collection:14


In [10]:
def retrieve_relevant_chunks(question, top_k = 3):
  """
  Given a user question, retrieve the most relevant document chunks from chromaDB.

  Parameters:
  question(str) : The user's question as a text string
  top_k(int) : Howmnay top results to return (default : 3)

  Returns:
    A dictinoary containing retrieved documents, distances, and metadata
  """
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings = [question_embedding],
      n_results = top_k
  )
  return results
print("Retrieval function defined successfully")
print("Function: retrieve_relevant_chunks(question, top_k = 3)")


Retrieval function defined successfully
Function: retrieve_relevant_chunks(question, top_k = 3)


In [11]:
test_question = "gen ai"
print(f"Test Question:{test_question}")
print("="*60)
results = retrieve_relevant_chunks(test_question, top_k =3)
print("\n Top 3 retrieved chunks:")
print("="*60)
for i, (doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
  print(f"\nResult{i+1}:")
  print(f"Subject: {meta['subject']}")
  print(f"Topic: {meta['topic']}")
  print(f"Distance: {dist:.4f}")
  print(f"Content: {doc[:120]}...")

Test Question:gen ai

 Top 3 retrieved chunks:

Result1:
Subject: Generative AI
Topic: Retrieval Augmented Generation
Distance: 1.1565
Content: RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowled...

Result2:
Subject: Generative AI
Topic: Prompt Engineering
Distance: 1.3246
Content: Prompt engineering is the practice of designing effective instructions or inputs for AI language models. A good prompt g...

Result3:
Subject: Generative AI
Topic: Large Language Models
Distance: 1.3476
Content: A Large Language Model or LLM is an AI model trained on massive amounts of text data. It can generate human-like text an...


In [12]:
def build_context_from_results(results):
  """
  Format chromaDB retrieval results into a readable context string.

  Parameters:
  results: The output from collection.query() - a dictionary.

  Returns:
  context_str (str): A formatted String of all retrieved document chunks
  """

  context_parts = []
  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
       chunk_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}]\n{doc}"
print("Context building function defined successfully")
print("Function: build_context_from_results(results)")

Context building function defined successfully
Function: build_context_from_results(results)


In [13]:
def generate_rag_answer(question, context):
  """
  Send the retrieved contex and question to the Groq LLM for answer generation.
  Parameters:
    question(str) : The user's question
    context(str) : The retrieved context chunks (formatted string)

  Returns:
    answer(str): The LLM's generated answer
  """
  system_prompt = """ You are a helpful academic assistant for engineering students.
  You will be ginven context retrived from a  college knowledge base, and a student's question.

RULES:
1. Use only the information available in the provided context.
2. If the answer is not found, respond exactly:
   "I don't have enough information in my knowledge base to answer this question."
3. Do not use external knowledge or make assumptions.
4. Provide clear, accurate, and beginner-friendly answers.
5. Cite the source when available.
  """
  user_prompt = f"""Context from Knowledge Base:
{context}
---
Student's Question:{question}
Please answer the question based only on the context provided above.
"""
  response = groq_client.chat.completions.create(
    model = "llama-3.1-8b-instant",
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature = 0.1,
    max_tokens = 500
  )
  answer = response.choices[0].message.content
  return answer
print("RAG generation function defined.")

RAG generation function defined.


In [14]:
def ask_college_assistant(question, top_k = 3, verbose = True):
  """
  Complete RAG pipeline : Given a question, retrieve relevant context and generate an answer.

  Parameters:
  question (str): The user's question
  top_k (int) : Number of chunks to retrieve (default : 3)
  verbose (bool) : Whether to print intermediate steps (default : True)

  Returns:
  answer (str) : The final generated answer
  """

  if verbose:
      print(f"Question: {question}")
      print("="*60)
      print("Step 1: Retrieving relevant documents...")

  results = retrieve_relevant_chunks(question, top_k = top_k)

  if verbose:
      print("Step 2: Building context from retrieved documents...")

  context = build_context_from_results(results)

  if verbose:
      print("Step 3: Generating answer with LLM...")

  answer = generate_rag_answer(question, context)

  if verbose:
      print("="*60)
      print("RAG Pipeline Completed.")

  return answer
print("Completed")

Completed
